In [1]:
import math

data = [
    ("Sunny",    "Hot",  "High",   "Weak",   "No"),
    ("Sunny",    "Hot",  "High",   "Strong", "No"),
    ("Overcast", "Hot",  "High",   "Weak",   "Yes"),
    ("Rain",     "Mild", "High",   "Weak",   "Yes"),
    ("Rain",     "Cool", "Normal", "Weak",   "Yes"),
    ("Rain",     "Cool", "Normal", "Strong", "No"),
    ("Overcast", "Cool", "Normal", "Strong", "Yes"),
    ("Sunny",    "Mild", "High",   "Weak",   "No"),
    ("Sunny",    "Cool", "Normal", "Weak",   "Yes"),
    ("Rain",     "Mild", "Normal", "Weak",   "Yes"),
    ("Sunny",    "Mild", "Normal", "Strong", "Yes"),
    ("Overcast", "Mild", "High",   "Strong", "Yes"),
    ("Overcast", "Hot",  "Normal", "Weak",   "Yes"),
    ("Rain",     "Mild", "High",   "Strong", "No"),
]

# ──────────────────────────────────────────────
# ÉTAPE 0 : Encoder les labels Yes/No → 1/0
# ──────────────────────────────────────────────
def encoder_labels(dataset):
    return [1 if row[4] == "Yes" else 0 for row in dataset]

# ──────────────────────────────────────────────
# ÉTAPE 1 : Prédiction initiale (moyenne des labels)
# C'est le F0 de XGBoost : on part d'une valeur de base
# ──────────────────────────────────────────────
def prediction_initiale(labels):
    return sum(labels) / len(labels)

# ──────────────────────────────────────────────
# ÉTAPE 2 : Calculer les résidus (gradients)
# résidu = label_réel - prédiction_actuelle
# C'est ce que le prochain arbre doit apprendre à corriger
# ──────────────────────────────────────────────
def calculer_residus(labels, predictions):
    return [y - p for y, p in zip(labels, predictions)]

# ──────────────────────────────────────────────
# ÉTAPE 3 : Calculer le score de similarité d'un nœud
# similarity = (somme des résidus)² / (nb_échantillons + lambda)
# lambda (régularisation) évite le surapprentissage
# ──────────────────────────────────────────────
def similarity_score(residus, lam=1.0):
    return (sum(residus) ** 2) / (len(residus) + lam)

# ──────────────────────────────────────────────
# ÉTAPE 4 : Calculer le gain d'un split sur une colonne
# gain = similarity(gauche) + similarity(droite) - similarity(parent)
# On choisit le split qui maximise ce gain
# ──────────────────────────────────────────────
def gain_split(dataset, residus, col, lam=1.0):
    valeurs = set(row[col] for row in dataset)  # Étape 4a : quelles valeurs existent dans cette colonne ?
    meilleur_gain = -1
    meilleur_val  = None

    sim_parent = similarity_score(residus, lam)  # Étape 4b : similarity du nœud parent

    for v in valeurs:
        indices_gauche = [i for i, row in enumerate(dataset) if row[col] == v]      # Étape 4c : indices à gauche (col == v)
        indices_droite = [i for i, row in enumerate(dataset) if row[col] != v]      # Étape 4d : indices à droite (col != v)

        res_gauche = [residus[i] for i in indices_gauche]  # Étape 4e : résidus gauche
        res_droite = [residus[i] for i in indices_droite]  # Étape 4f : résidus droite

        if not res_gauche or not res_droite:
            continue

        g = similarity_score(res_gauche, lam) + similarity_score(res_droite, lam) - sim_parent  # Étape 4g : gain du split
        print(f"  Gain(col={col}, val={v}) = {g:.4f}")

        if g > meilleur_gain:
            meilleur_gain = g
            meilleur_val  = v

    return meilleur_val, meilleur_gain

# ──────────────────────────────────────────────
# ÉTAPE 5 : Choisir le meilleur split (colonne + valeur)
# On teste toutes les colonnes et on garde celle avec le gain max
# ──────────────────────────────────────────────
def meilleur_split(dataset, residus, attributs, lam=1.0):
    meilleur_gain = -1
    meilleur_col  = None
    meilleur_val  = None

    for col in attributs:
        val, g = gain_split(dataset, residus, col, lam)
        if g > meilleur_gain:
            meilleur_gain = g
            meilleur_col  = col
            meilleur_val  = val

    return meilleur_col, meilleur_val

# ──────────────────────────────────────────────
# ÉTAPE 6 : Construire un arbre de régression sur les résidus
# Chaque feuille contient la valeur de sortie = somme(résidus) / (nb + lambda)
# ──────────────────────────────────────────────
def construire_arbre(dataset, residus, attributs, profondeur_max=2, profondeur=0, lam=1.0):
    # ── Cas 1 : profondeur max atteinte → feuille
    if profondeur >= profondeur_max or len(attributs) == 0:
        valeur_feuille = sum(residus) / (len(residus) + lam)
        return {"feuille": valeur_feuille}

    # ── Cas 2 : choisir le meilleur split
    col, val = meilleur_split(dataset, residus, attributs, lam)

    if col is None:
        valeur_feuille = sum(residus) / (len(residus) + lam)
        return {"feuille": valeur_feuille}

    # ── Cas 3 : partitionner et appeler récursivement
    indices_gauche = [i for i, row in enumerate(dataset) if row[col] == val]   # Étape 6a : indices gauche (col == val)
    indices_droite = [i for i, row in enumerate(dataset) if row[col] != val]   # Étape 6b : indices droite (col != val)

    subset_gauche  = [dataset[i] for i in indices_gauche]
    subset_droite  = [dataset[i] for i in indices_droite]
    res_gauche     = [residus[i] for i in indices_gauche]
    res_droite     = [residus[i] for i in indices_droite]

    attributs_restants = [a for a in attributs if a != col]

    arbre = {
        "col": col,
        "val": val,
        "gauche": construire_arbre(subset_gauche, res_gauche, attributs_restants, profondeur_max, profondeur + 1, lam),
        "droite": construire_arbre(subset_droite, res_droite, attributs_restants, profondeur_max, profondeur + 1, lam),
    }
    return arbre

# ──────────────────────────────────────────────
# ÉTAPE 7 : Prédire la correction d'un seul exemple avec un arbre
# ──────────────────────────────────────────────
def predire_arbre(arbre, row):
    # ── Cas feuille : retourner la valeur
    if "feuille" in arbre:
        return arbre["feuille"]

    # ── Sinon : aller à gauche ou droite selon la valeur de la colonne
    if row[arbre["col"]] == arbre["val"]:
        return predire_arbre(arbre["gauche"], row)
    else:
        return predire_arbre(arbre["droite"], row)

# ──────────────────────────────────────────────
# ÉTAPE 8 : Algorithme XGBoost complet
# On construit N arbres, chaque arbre corrige les erreurs du précédent
# ──────────────────────────────────────────────
def xgboost(dataset, attributs, n_arbres=3, lr=0.3, profondeur_max=2, lam=1.0):
    labels      = encoder_labels(dataset)                  # Étape 8a : encoder Yes/No → 1/0
    predictions = [prediction_initiale(labels)] * len(dataset)  # Étape 8b : prédiction de départ (F0)
    arbres      = []

    for t in range(n_arbres):
        print(f"\n── Arbre {t+1} ──")
        residus = calculer_residus(labels, predictions)    # Étape 8c : calculer les résidus
        arbre   = construire_arbre(dataset, residus, attributs, profondeur_max, lam=lam)  # Étape 8d : apprendre l'arbre sur les résidus
        arbres.append(arbre)

        # Étape 8e : mettre à jour les prédictions
        for i, row in enumerate(dataset):
            predictions[i] += lr * predire_arbre(arbre, row)

    return predictions, arbres

# ──────────────────────────────────────────────
# ÉTAPE 9 : Convertir les prédictions en Yes/No
# ──────────────────────────────────────────────
def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def classer(predictions):
    return ["Yes" if sigmoid(p) >= 0.5 else "No" for p in predictions]

# ──────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────
attributs = [0, 1, 2, 3]  # Outlook, Temperature, Humidity, Wind

predictions, arbres = xgboost(data, attributs, n_arbres=3, lr=0.3, profondeur_max=2)

labels_reels   = [row[4] for row in data]
labels_predits = classer(predictions)

print("\n── Résultats ──")
print(f"{'Réel':<8} {'Prédit':<8} {'OK?'}")
print("-" * 24)
correct = 0
for reel, predit in zip(labels_reels, labels_predits):
    ok = "✓" if reel == predit else "✗"
    if reel == predit:
        correct += 1
    print(f"{reel:<8} {predit:<8} {ok}")

print(f"\nPrécision : {correct}/{len(data)} = {correct/len(data)*100:.1f}%")


── Arbre 1 ──
  Gain(col=0, val=Rain) = 0.0122
  Gain(col=0, val=Overcast) = 0.5937
  Gain(col=0, val=Sunny) = 0.3932
  Gain(col=1, val=Cool) = 0.0534
  Gain(col=1, val=Mild) = 0.0052
  Gain(col=1, val=Hot) = 0.0950
  Gain(col=2, val=Normal) = 0.5625
  Gain(col=2, val=High) = 0.5625
  Gain(col=3, val=Weak) = 0.1866
  Gain(col=3, val=Strong) = 0.1866
  Gain(col=1, val=Cool) = -0.0574
  Gain(col=1, val=Mild) = -0.0574
  Gain(col=1, val=Hot) = -0.0680
  Gain(col=2, val=Normal) = -0.0680
  Gain(col=2, val=High) = -0.0680
  Gain(col=3, val=Weak) = -0.0680
  Gain(col=3, val=Strong) = -0.0680
  Gain(col=1, val=Cool) = 0.0970
  Gain(col=1, val=Mild) = 0.0679
  Gain(col=1, val=Hot) = 0.3678
  Gain(col=2, val=Normal) = 0.7345
  Gain(col=2, val=High) = 0.7345
  Gain(col=3, val=Weak) = 0.3113
  Gain(col=3, val=Strong) = 0.3113

── Arbre 2 ──
  Gain(col=0, val=Rain) = 0.0048
  Gain(col=0, val=Overcast) = 0.3613
  Gain(col=0, val=Sunny) = 0.2583
  Gain(col=1, val=Cool) = 0.0164
  Gain(col=1, val=Mi